## 2016年小时级数据

转化npz格式，24h为一个时间窗口滑动

### 去空值
 
24h周期对其插值法

In [11]:
import pandas as pd

# 读取数据
file_path = "electricityLondon/electricityLondon.csv"
df = pd.read_csv(file_path)
first_col = df.columns[0]
df = df.set_index(first_col)


print("data shape:", df.shape)
print("=" * 50)

# 检查是否存在缺失值
if not df.isnull().values.any():
    print("✅ no missing value")
else:
    print("⚠️ spot missing value\n")
    
    # 找出有缺失值的行
    missing_rows = df[df.isnull().any(axis=1)]
    
    for idx, row in missing_rows.iterrows():
        missing_cols = row[row.isnull()].index.tolist()
        print(f"第 {idx} 行存在缺失值")
        print("缺失列名:", missing_cols)
        print("-" * 50)

data shape: (8784, 68)
⚠️ spot missing value

第 2016/2/4 9:00 行存在缺失值
缺失列名: ['Robin_office_Soledad']
--------------------------------------------------
第 2016/2/5 5:00 行存在缺失值
缺失列名: ['Mouse_health_Modesto']
--------------------------------------------------
第 2016/2/5 6:00 行存在缺失值
缺失列名: ['Mouse_health_Modesto']
--------------------------------------------------
第 2016/3/5 11:00 行存在缺失值
缺失列名: ['Mouse_health_Ileana']
--------------------------------------------------
第 2016/3/5 12:00 行存在缺失值
缺失列名: ['Mouse_health_Ileana']
--------------------------------------------------
第 2016/3/5 13:00 行存在缺失值
缺失列名: ['Mouse_health_Ileana']
--------------------------------------------------
第 2016/3/5 14:00 行存在缺失值
缺失列名: ['Mouse_health_Ileana']
--------------------------------------------------
第 2016/3/5 15:00 行存在缺失值
缺失列名: ['Mouse_health_Ileana']
--------------------------------------------------
第 2016/3/5 16:00 行存在缺失值
缺失列名: ['Mouse_health_Ileana']
--------------------------------------------------
第 2016/3/

In [12]:
# ==========================================
# 周期对齐插值填补缺失值
# ==========================================
import numpy as np

PERIOD = 24  # 每天24小时

total_missing = df.isnull().sum().sum()

if total_missing > 0:
    print(f"\n开始使用周期对齐插值填补缺失值 (period={PERIOD})")
    
    for col_idx, col_name in enumerate(df.columns):
        series = df[col_name].values.astype(float)
        nan_idx = np.where(np.isnan(series))[0]

        if len(nan_idx) == 0:
            continue

        print(f"\n列 `{col_name}` (第 {col_idx+1} 列)")
        print("缺失位置:", nan_idx.tolist())

        for t in nan_idx:
            # 检查是否能做周期对齐
            if t - PERIOD >= 0 and t + PERIOD < len(series):
                series[t] = 0.5 * (series[t - PERIOD] + series[t + PERIOD])
            else:
                print(f"⚠️ 位置 {t} 超出周期范围，无法周期插值")

        df[col_name] = series

    print("\n周期插值完成！")

    # 再检查一次
    remaining_missing = df.isnull().sum().sum()
    print(f"剩余缺失值数量: {remaining_missing}")

    # 保存新文件
    output_path = "electricityLondon/electricityLondon_24hfilled.csv"
    df_reset = df.reset_index()
    df_reset.to_csv(output_path, index=False)
    #df.to_csv(output_path, index=False)
    print("已保存为:", output_path)

else:
    print("\n无需插值，数据本身无缺失值。")


开始使用周期对齐插值填补缺失值 (period=24)

列 `Robin_office_Soledad` (第 42 列)
缺失位置: [825, 4815]

列 `Mouse_health_Buddy` (第 62 列)
缺失位置: [5998, 6042, 6106, 6163, 6209, 6267, 6326, 6375, 6440, 6502, 6554, 6635, 6685, 6733, 6803, 6851, 6899, 6970, 7018, 7066, 7136, 7185, 7236, 7310, 7359, 7407, 7481, 7526, 7571, 7636, 7682, 7725, 7789, 7841, 7936, 8001, 8034, 8076, 8146, 8187, 8231, 8301, 8350, 8397, 8463, 8513, 8559, 8618, 8700, 8756]

列 `Mouse_health_Modesto` (第 63 列)
缺失位置: [845, 846, 2030, 2031]

列 `Mouse_health_Ileana` (第 66 列)
缺失位置: [1547, 1548, 1549, 1550, 1551, 1552, 1553, 1554, 1555, 1977, 1979]

周期插值完成！
剩余缺失值数量: 0
已保存为: electricityLondon/electricityLondon_24hfilled.csv


### 模仿化为npz格式

In [14]:
import pandas as pd
import numpy as np

# ==========================================
# 参数
# ==========================================
INPUT_CSV = "electricityLondon/electricityLondon_24hfilled.csv"
OUTPUT_NPZ = "electricityLondon/electricityLondon.npz"

T_IN = 12
T_OUT = 12

# ==========================================
# 1. 读取 CSV
# ==========================================
print("读取 CSV:", INPUT_CSV)

df = pd.read_csv(INPUT_CSV)

# 第一列是时间索引
time_col = df.columns[0]
df[time_col] = pd.to_datetime(df[time_col])
df = df.set_index(time_col)

print("数据形状:", df.shape)

num_nodes = df.shape[1]
total_timesteps = df.shape[0]

print("总时间步:", total_timesteps)
print("节点数量:", num_nodes)

# ==========================================
# 2. 构造时间特征（一天中的位置）
# ==========================================
# hour / 24，保留8位小数
time_feature = (df.index.hour / 24.0).to_numpy()
time_feature = np.round(time_feature, 8)

# ==========================================
# 3. 计算样本数量
# ==========================================
num_samples = total_timesteps - T_IN - T_OUT + 1
print("可生成样本数:", num_samples)

# ==========================================
# 4. 初始化数组
# ==========================================
x = np.zeros((num_samples, T_IN, num_nodes, 3), dtype=np.float64)
y = np.zeros((num_samples, T_OUT, num_nodes, 1), dtype=np.float64)

data_values = df.values  # (T, N)

# ==========================================
# 5. 构造滑动窗口
# ==========================================
for i in range(num_samples):
    
    # 输入窗口
    x_window = data_values[i : i + T_IN]                # (12, N)
    x_time = time_feature[i : i + T_IN]                 # (12,)
    
    # 赋值
    x[i, :, :, 0] = x_window                            # 真实值
    x[i, :, :, 1] = x_time[:, None]                     # 时间特征
    x[i, :, :, 2] = 0.0                                 # 占位符
    
    # 输出窗口
    y_window = data_values[i + T_IN : i + T_IN + T_OUT] # (12, N)
    y[i, :, :, 0] = y_window

# ==========================================
# 6. 构造 offsets
# ==========================================
x_offsets = np.arange(-T_IN + 1, 1).reshape(-1, 1)
y_offsets = np.arange(1, T_OUT + 1).reshape(-1, 1)

print("x shape:", x.shape)
print("y shape:", y.shape)
print("x_offsets:", x_offsets.flatten())
print("y_offsets:", y_offsets.flatten())

# ==========================================
# 7. 保存为 npz
# ==========================================
np.savez(
    OUTPUT_NPZ,
    x=x,
    y=y,
    x_offsets=x_offsets.astype(np.int64),
    y_offsets=y_offsets.astype(np.int64),
)

print("保存完成:", OUTPUT_NPZ)

读取 CSV: electricityLondon/electricityLondon_24hfilled.csv
数据形状: (8784, 68)
总时间步: 8784
节点数量: 68
可生成样本数: 8761
x shape: (8761, 12, 68, 3)
y shape: (8761, 12, 68, 1)
x_offsets: [-11 -10  -9  -8  -7  -6  -5  -4  -3  -2  -1   0]
y_offsets: [ 1  2  3  4  5  6  7  8  9 10 11 12]
保存完成: electricityLondon/electricityLondon.npz


In [16]:
# ==========================================
# 8. 保存列名 -> 节点维度 映射文件
# ==========================================

MAPPING_CSV = "electricityLondon/node_mapping.csv"

# df.columns 就是当前节点顺序
node_mapping = pd.DataFrame({
    "node_index": np.arange(len(df.columns)),
    "column_name": df.columns
})

node_mapping.to_csv(MAPPING_CSV, index=False)

print("节点映射文件已保存:", MAPPING_CSV)

节点映射文件已保存: electricityLondon/node_mapping.csv


### 划分train test val

In [15]:
import numpy as np

# ===============================
# 1. 读取完整数据
# ===============================
INPUT_NPZ = "electricityLondon/electricityLondon.npz"

data = np.load(INPUT_NPZ)

x = data["x"]
y = data["y"]
x_offsets = data["x_offsets"]
y_offsets = data["y_offsets"]

total_samples = x.shape[0]
print("总样本数:", total_samples)

# ===============================
# 2. 计算划分边界（60/20/20）
# ===============================
train_size = int(total_samples * 0.6)
val_size = int(total_samples * 0.2)
test_size = total_samples - train_size - val_size

print("train:", train_size)
print("val:", val_size)
print("test:", test_size)

# ===============================
# 3. 顺序划分
# ===============================
x_train = x[:train_size]
y_train = y[:train_size]

x_val = x[train_size:train_size+val_size]
y_val = y[train_size:train_size+val_size]

x_test = x[train_size+val_size:]
y_test = y[train_size+val_size:]

# ===============================
# 4. 保存为三个 npz 文件
# ===============================
np.savez(
    "electricityLondon/train.npz",
    x=x_train,
    y=y_train,
    x_offsets=x_offsets,
    y_offsets=y_offsets,
)

np.savez(
    "electricityLondon/val.npz",
    x=x_val,
    y=y_val,
    x_offsets=x_offsets,
    y_offsets=y_offsets,
)

np.savez(
    "electricityLondon/test.npz",
    x=x_test,
    y=y_test,
    x_offsets=x_offsets,
    y_offsets=y_offsets,
)

print("划分完成并保存！")

总样本数: 8761
train: 5256
val: 1752
test: 1753
划分完成并保存！
